# BB8 — Phase 5 & 6: Training Demo & Text Generation

This notebook demonstrates:

1. **Quick model training** on Tiny Shakespeare (small model, ~5 minutes)
2. **Training curve visualisation** — loss and perplexity over time
3. **Text generation** with all four decoding strategies
4. **Effect of temperature** on generation quality

> This notebook trains a small model from scratch for demonstration.  For better quality, use `train.py` with the medium model config.

In [ ]:
import sys, os, json, math
sys.path.insert(0, os.path.dirname(os.getcwd()))

import torch
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 4)

from torch.utils.data import DataLoader
from tokenizer import CharTokenizer
from datasets.text_dataset import TextDataset, create_train_val_split
from models.language_model import BB8LM
from training.trainer import Trainer
from inference.generator import TextGenerator
from evaluation.evaluator import Evaluator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Load Data & Build Tokenizer

In [ ]:
DATA_PATH = '../data/tiny_shakespeare.txt'

try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f'Loaded {len(text):,} characters')
except FileNotFoundError:
    # Fallback — use a small snippet for demonstration
    text = """
HAMLET: To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them.
""".strip() * 200
    print(f'Using sample text ({len(text):,} characters)')

tokenizer = CharTokenizer()
tokenizer.train([text])
print(tokenizer)

## 2. Build Dataset & DataLoaders

In [ ]:
SEQ_LEN = 128
BATCH_SIZE = 64

dataset = TextDataset(text, tokenizer, seq_len=SEQ_LEN)
train_ds, val_ds = create_train_val_split(dataset, val_fraction=0.1)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 3. Build Model

In [ ]:
torch.manual_seed(42)

model = BB8LM(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=64,
    num_layers=2,
    num_heads=2,
    d_ff=256,
    max_seq_len=SEQ_LEN,
    dropout=0.1,
)

print(f'Parameters: {model.count_parameters():,}')

## 4. Train

In [ ]:
EPOCHS = 5

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-3,
    warmup_steps=50,
    checkpoint_dir='../checkpoints/notebook_demo',
    log_interval=20,
    eval_interval=100,
    save_interval=9999,   # disable auto-save during notebook demo
    device=DEVICE,
)

history = trainer.train(num_epochs=EPOCHS)

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Per-step training loss (smoothed)
losses = trainer.train_loss_history
window = max(1, len(losses) // 100)
smoothed = [sum(losses[max(0,i-window):i+1]) / min(i+1, window) for i in range(len(losses))]

axes[0].plot(losses, alpha=0.3, color='#4e79a7', linewidth=0.5, label='Raw')
axes[0].plot(smoothed, color='#4e79a7', linewidth=2, label='Smoothed')
axes[0].set_title('Training Loss (per step)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend()

# Per-epoch train + val loss
epochs_x = range(1, len(history['train_loss']) + 1)
axes[1].plot(epochs_x, history['train_loss'], 'o-', color='#4e79a7', label='Train')
if history['val_loss']:
    axes[1].plot(epochs_x[:len(history['val_loss'])], history['val_loss'], 
                 's-', color='#e15759', label='Val')
axes[1].set_title('Loss per Epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

# Perplexity
if history['perplexity']:
    ppl_x = range(1, len(history['perplexity']) + 1)
    axes[2].plot(ppl_x, history['perplexity'], 'o-', color='#f28e2b')
    axes[2].set_title('Validation Perplexity per Epoch')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Perplexity (lower is better)')
    axes[2].axhline(y=tokenizer.get_vocab_size(), color='gray', linestyle='--', 
                    label='Random baseline', alpha=0.5)
    axes[2].legend()

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/training_curves.png', dpi=150)
plt.show()
print('Saved: outputs/training_curves.png')

## 6. Text Generation — All Strategies

In [ ]:
generator = TextGenerator(model, tokenizer, device=DEVICE)

PROMPT = 'To be or not to be'
MAX_TOKENS = 200

results = generator.compare_strategies(
    prompt=PROMPT,
    max_new_tokens=MAX_TOKENS,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.1,
)

## 7. Effect of Temperature

In [ ]:
print('Temperature Sweep (Top-P sampling, p=0.9)')
print('='*60)

temperatures = [0.3, 0.7, 1.0, 1.3]

for temp in temperatures:
    text_out = generator.generate(
        PROMPT, max_new_tokens=80, strategy='top_p',
        temperature=temp, top_p=0.9, repetition_penalty=1.0
    )
    print(f'\n[temp={temp}]')
    print(text_out)
    print('-'*40)

## 8. Final Evaluation

In [ ]:
evaluator = Evaluator(model, tokenizer, device=DEVICE)

val_metrics = evaluator.evaluate_dataset(val_loader)

print('Validation Metrics')
print('='*40)
for key, val in val_metrics.items():
    print(f'  {key:20s}: {val:.4f}')

print(f'\n  Vocab size (random baseline PPL): {tokenizer.get_vocab_size()}')
print(f'  Achieved perplexity reduction:  {tokenizer.get_vocab_size() / val_metrics["perplexity"]:.1f}x better than random')